In [1]:
import re
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, make_scorer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


# Исследование и прогнозирование качества блюд (Food & Nutrition)

## 1. Подготовка и фильтрация данных (Feature Selection)

**Физика задачи:** Исходная таблица  содержит 680 признаков, среди которых перемешаны ингредиенты, географические теги (, ), кулинарные категории (, ), макронутриенты (, ) и способы термической обработки (, , ).

Для обучения модели, способной оценивать блюдо исключительно по продуктовой корзине пользователя, необходимо отфильтровать все «не-ингредиентные» колонки. Для этого мы используем поваренную книгу , где в поле  авторы зафиксировали только реальные продукты.

### 1.1. Загрузка исходных данных

Загружаем структурированный JSON-архив и плоскую таблицу признаков.

In [2]:
with open("../datasets/full_format_recipes.json", "r", encoding="utf-8") as f:
    recipes_data = json.load(f)

df_raw = pd.read_csv("../datasets/epi_r.csv")

print(f"Всего рецептов в JSON: {len(recipes_data)}")
print(f"Размер исходной таблицы: {df_raw.shape} (строк, колонок)")


### 1.2. Формирование кулинарного словаря (White-list ингредиентов)

Извлекаем все слова из списков ингредиентов рецептов в JSON, формируя словарь слов, которые реально используются в кулинарии.

In [ ]:
ingredient_words = set()
for recipe in recipes_data:
    ingredients = recipe.get("ingredients")
    if ingredients:
        for item in ingredients:
            words = re.findall(r"\b[a-z]+\b", item.lower())
            ingredient_words.update(words)

print(f"Уникальных кулинарных слов в базе рецептов: {len(ingredient_words)}")


### 1.3. Фильтрация признаков и приведение имен колонок к стандарту пользовательского ввода

1. Приводим составные имена ( -> ,  -> ) к стандартам пользовательского ввода и технического задания.
2. Исключаем из обучающей выборки черный список тегов: географию, макронутриенты, категории и способы приготовления.
3. Оставляем только те колонки, чьи названия полностью состоят из реальных кулинарных слов.

In [4]:
# 1. Алиасы к стандарту пользовательского ввода
df = df_raw.rename(columns={"milk/cream": "milk", "jam or jelly": "jam"}).copy()

# 2. Черный список (не-ингредиентные признаки)
blacklist = {
    "title", "calories", "protein", "fat", "sodium",
    "leftovers", "california", "dominican republic",
    "low cholesterol", "pan-fry", "fruit", "stew", "braise"
}

# 3. Фильтрация колонок
clean_cols = []
for col in df.columns:
    if col == "rating":
        clean_cols.append(col)
        continue
    if col in blacklist:
        continue
    words = re.findall(r"[a-z]+", col.lower())
    if words and all(w in ingredient_words for w in words):
        clean_cols.append(col)

df_clean = df[clean_cols].copy()

print(f"Исходных колонок: {df.shape[1]}")
print(f"Чистых колонок (ингредиенты + rating): {df_clean.shape[1]}")

# 4. Контрольная валидация состава признаков
assert "title" not in df_clean.columns
assert "protein" not in df_clean.columns
assert "fruit" not in df_clean.columns
assert "leftovers" not in df_clean.columns
assert "california" not in df_clean.columns
assert "pan-fry" not in df_clean.columns
assert "milk" in df_clean.columns
assert "jam" in df_clean.columns
print(" Фильтрация признаков успешно завершена!")


### 1.4. Обработка пропусков и удаление дубликатов

Проверяем наличие пропусков в целевой переменной  и признаках, а также удаляем строки-дубликаты для предотвращения утечки данных.

In [5]:
# Удаляем строки без таргета (если есть)
df_clean = df_clean.dropna(subset=["rating"])

# Удаление дубликатов
duplicates_before = df_clean.duplicated().sum()
if duplicates_before > 0:
    df_clean = df_clean.drop_duplicates()

print(f"Удалено строк-дубликатов: {duplicates_before}")
print(f"Итоговый размер обучающей матрицы: {df_clean.shape}")


### 1.5. Экспорт списка чистых ингредиентов

Сохраняем список из 497 эталонных ингредиентов в CSV-файл для параллельного модуля работы с USDA API.

In [6]:
ingredient_list = [c for c in df_clean.columns if c != "rating"]
pd.Series(ingredient_list).to_csv("../datasets/clean_ingredients.csv", index=False, header=["ingredient"])
print(f"Список из {len(ingredient_list)} ингредиентов сохранен в datasets/clean_ingredients.csv")


## 2. Обучение моделей: Регрессия


### 2.1. Формирование обучающей и тестовой выборок, расчет Baseline

Отделяем признаки-ингредиенты от целевой оценки rating.

In [38]:
X = df_clean.drop(columns=['rating'])
y = df_clean['rating']

Делим данные на train (80%) и test (20%) со случайным сидом:

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Размер обучающей выборки (X_train): {X_train.shape}")
print(f"Размер тестовой выборки  (X_test):  {X_test.shape}")

Размер обучающей выборки (X_train): (14546, 497)
Размер тестовой выборки  (X_test):  (3637, 497)


Рассчитываем наивный baseline (прогноз среднего значения rating по обучающей выборке):

In [40]:
naive_mean = y_train.mean()

Формируем массив предсказаний наивного алгоритма для теста:

In [41]:
y_pred_naive = np.full_like(y_test, fill_value=naive_mean)

Рассчитываем и фиксируем RMSE наивного алгоритма:

In [42]:
rmse_naive = np.sqrt(mean_squared_error(y_test, y_pred_naive))
print(f"\nСредний рейтинг по базе:  {naive_mean:.4f}")
print(f"RMSE наивного алгоритма: {rmse_naive:.4f}")


Средний рейтинг по базе:  3.7157
RMSE наивного алгоритма: 1.2993


### 2.2. Сравнение базовых моделей регрессии

**Цель:** Оценить базовые алгоритмы регрессии на 5-фолдовой кросс-валидации по метрике RMSE и сравнить их с наивным baseline ().

1. **Linear Regression** (линейная регрессия без регуляризации)
2. **Ridge Regression** (линейная регрессия с L2-регуляризацией)
3. **Decision Tree Regressor** (решающее дерево)

In [43]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42)
}

In [37]:
print(f"Базовый ориентир (Наивный RMSE): {rmse_naive:.4f}\n")
print("Результаты на кросс-валидации (5 фолдов):")

cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, 
        cv=5, 
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    rmse_cv = -scores.mean()
    cv_scores[name] = rmse_cv
    print(f"- {name}: RMSE = {rmse_cv:.4f} (+/- {scores.std():.4f})")

Базовый ориентир (Наивный RMSE): 1.2993

Результаты на кросс-валидации (5 фолдов):


- Linear Regression: RMSE = 1.2622 (+/- 0.0262)
- Ridge Regression: RMSE = 1.2561 (+/- 0.0244)
- Decision Tree: RMSE = 1.2933 (+/- 0.0281)


### 2.3. Подбор гиперпараметров для лучшей базовой модели (GridSearchCV)

Среди одиночных моделей лидером стала **Ridge Regression** ().
Оптимизируем коэффициент регуляризации  через  на 5 фолдах кросс-валидации.

In [44]:
# Задаем диапазон значений alpha для проверки
param_grid_ridge = {
    'alpha': [0.1, 1.0, 10.0, 50.0, 100.0, 200.0, 500.0]
}

In [45]:
grid_ridge = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid_ridge,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_ridge.fit(X_train, y_train)

best_alpha = grid_ridge.best_params_['alpha']
best_ridge_rmse = -grid_ridge.best_score_

print(f"Лучший параметр alpha: {best_alpha}")
print(f"RMSE Ridge после тюнинга: {best_ridge_rmse:.4f} (было 1.2561)")

Лучший параметр alpha: 50.0
RMSE Ridge после тюнинга: 1.2428 (было 1.2561)


### 2.4. Исследование ансамблей моделей (Ensemble Methods)

**Цель:** Проверить, сможет ли консилиум моделей превзойти лучшую одиночную модель (, ).

1. **Random Forest Regressor** (бэггинг)
2. **Gradient Boosting Regressor** (бустинг)

In [46]:
ensembles = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

In [47]:
print("Результаты ансамблей на кросс-валидации (5 фолдов):")

ensemble_scores = {}
for name, model in ensembles.items():
    scores = cross_val_score(
        model, X_train, y_train, 
        cv=5, 
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    rmse_cv = -scores.mean()
    ensemble_scores[name] = rmse_cv
    print(f"- {name}: RMSE = {rmse_cv:.4f} (+/- {scores.std():.4f})")

Результаты ансамблей на кросс-валидации (5 фолдов):
- Random Forest: RMSE = 1.2506 (+/- 0.0222)
- Gradient Boosting: RMSE = 1.2399 (+/- 0.0191)


### 2.5. Финальная оценка лучших моделей регрессии на тестовой выборке

Оцениваем на скрытой тестовой выборке только двух победителей предварительного отбора:
1. **Ridge** ()
2. **Gradient Boosting**

In [48]:
# 1. Обучаем лучшую модель Ridge на всем train и делаем предикт на test
best_ridge = Ridge(alpha=best_alpha)
best_ridge.fit(X_train, y_train)
y_pred_ridge = best_ridge.predict(X_test)
test_rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))# 1. Обучаем лучшую модель Ridge на всем train и делаем предикт на test
best_ridge = Ridge(alpha=best_alpha)
best_ridge.fit(X_train, y_train)
y_pred_ridge = best_ridge.predict(X_test)
test_rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

In [49]:
# 2. Обучаем лучший ансамбль Gradient Boosting на всем train и делаем предикт на test
best_gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
best_gb.fit(X_train, y_train)
y_pred_gb = best_gb.predict(X_test)
test_rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

In [52]:
# 3. Сводная таблица результатов
results_df = pd.DataFrame({
    'Модель': ['Наивный Baseline', 'Ridge (alpha=50.0)', 'Gradient Boosting'],
    'Test RMSE': [rmse_naive, test_rmse_ridge, test_rmse_gb],
    'Улучшение относительно Baseline': [
        '0.00%',
        f"{(rmse_naive - test_rmse_ridge) / rmse_naive * 100:.2f}%",
        f"{(rmse_naive - test_rmse_gb) / rmse_naive * 100:.2f}%"
    ]
})
print("ИТОГИ ЭТАПА РЕГРЕССИИ (НА ТЕСТЕ):")
display(results_df)

ИТОГИ ЭТАПА РЕГРЕССИИ (НА ТЕСТЕ):


,Модель,Test RMSE,Улучшение относительно Baseline
0,Наивный Baseline,1.299324,0.00%
1,Ridge (alpha=50.0),1.210233,6.86%
2,Gradient Boosting,1.203969,7.34%


## 3. Обучение моделей: Классификация рейтингов

Переходим от непрерывного предсказания к задаче категоризации рецептов.

### 3.1. Классификация на 6 целых классов (0, 1, 2, 3, 4, 5)

In [53]:
# 1. Округляем рейтинги до ближайшего целого
y_int = np.round(df_clean['rating']).astype(int)

# 2. Делим на train и test со стратификацией по классам
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)

# 3. Смотрим распределение классов в обучающей выборке
print("Распределение классов в train:")
print(pd.Series(y_train_c).value_counts().sort_index())

Распределение классов в train:
rating
0    1294
1     116
2     472
3    1084
4    9680
5    1900
Name: count, dtype: int64


#### 3.1.1. Наивный классификатор (Baseline по Accuracy)

Самым частым классом в обучающей выборке является оценка `4`.  
Наивный алгоритм работает по правилу мажоритарного класса: на любой рецепт из тестовой выборки он отвечает оценкой `4`.  
Рассчитаем долю правильных ответов (Accuracy) этого наивного подхода, чтобы получить планку, которую должны превзойти реальные модели.

In [54]:
# Находим самый частый класс (моду) в обучающей выборке
most_frequent_class = y_train_c.mode()[0]

# Наивный робот ставит этот класс всем рецептам из test
y_pred_naive_c = np.full_like(y_test_c, fill_value=most_frequent_class)

# Считаем точность (Accuracy)
acc_naive_c = accuracy_score(y_test_c, y_pred_naive_c)

print(f"Самый частый класс: {most_frequent_class}")
print(f"Accuracy наивного классификатора: {acc_naive_c:.4f} ({acc_naive_c * 100:.2f}%)")

Самый частый класс: 4
Accuracy наивного классификатора: 0.6654 (66.54%)


#### 3.1.2. Обучение базовых моделей классификации (6 классов)

**Цель:** Проверить, способны ли алгоритмы классификации превзойти наивный baseline (`66.54%`) на задаче с 6 классами (`0, 1, 2, 3, 4, 5`).

Сравниваем три принципиально разных классификатора:
1. **Logistic Regression:** линейный вероятностный классификатор с L2-регуляризацией.
2. **Decision Tree Classifier:** дерево решений, разделяющее пространство ингредиентов на классы по критерию Джини (Gini impurity).
3. **K-Nearest Neighbors (KNN):** классификация по голосованию ближайших соседей в пространстве ингредиентов.

*Сравнение проводим на кросс-валидации (5 фолдов) по метрике Accuracy.*

In [56]:
models_c6 = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

In [57]:
print(f"Базовый ориентир (Наивная Accuracy): {acc_naive_c:.4f}\n")
print("Результаты моделей на кросс-валидации (5 фолдов):")

cv_c6_scores = {}
for name, model in models_c6.items():
    scores = cross_val_score(
        model, X_train_c, y_train_c, 
        cv=5, 
        scoring='accuracy',
        n_jobs=-1
    )
    mean_acc = scores.mean()
    cv_c6_scores[name] = mean_acc
    print(f"- {name}: Accuracy = {mean_acc:.4f} (+/- {scores.std():.4f})")

Базовый ориентир (Наивная Accuracy): 0.6654

Результаты моделей на кросс-валидации (5 фолдов):
- Logistic Regression: Accuracy = 0.6767 (+/- 0.0014)
- Decision Tree: Accuracy = 0.6694 (+/- 0.0036)
- KNN: Accuracy = 0.6002 (+/- 0.0082)


#### 3.1.3. Подбор гиперпараметров для лучшей модели (GridSearchCV)

Лидером среди базовых классификаторов стала **Logistic Regression** с точностью `67.67%`.  
Оптимизируем её гиперпараметр регуляризации `C` (сила штрафа) с помощью `GridSearchCV` на 5 фолдах кросс-валидации.

In [58]:
param_grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0]
}

In [59]:
grid_lr = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train_c, y_train_c)

best_c = grid_lr.best_params_['C']
best_lr_cv_acc = grid_lr.best_score_

print(f"Лучший параметр C: {best_c}")
print(f"Accuracy после тюнинга на CV: {best_lr_cv_acc:.4f} ({best_lr_cv_acc * 100:.2f}%)")

Лучший параметр C: 0.1
Accuracy после тюнинга на CV: 0.6789 (67.89%)


#### 3.1.4. Оценка лучшей модели (6 классов) на тестовой выборке

Оцениваем победителя первого этапа — **Logistic Regression (C=0.1)** — на тестовой выборке `X_test_c` и сравниваем её Accuracy с наивным baseline (`66.54%`).

In [60]:
# Обучаем лучшую модель с C=0.1 на всем train
best_lr_c6 = LogisticRegression(C=best_c, max_iter=1000, random_state=42)
best_lr_c6.fit(X_train_c, y_train_c)

# Делаем предсказание на тесте
y_pred_lr_c6 = best_lr_c6.predict(X_test_c)
test_acc_lr_c6 = accuracy_score(y_test_c, y_pred_lr_c6)

print(f"Test Accuracy (Logistic Regression C={best_c}): {test_acc_lr_c6:.4f} ({test_acc_lr_c6 * 100:.2f}%)")

Test Accuracy (Logistic Regression C=0.1): 0.6800 (68.00%)


In [61]:
results_c6 = pd.DataFrame({
    'Модель (6 классов)': ['Наивный Baseline (класс 4)', f'Logistic Regression (C={best_c})'],
    'Test Accuracy': [acc_naive_c, test_acc_lr_c6]
})

results_c6

,Модель (6 классов),Test Accuracy
0,Наивный Baseline (класс 4),0.665384
1,Logistic Regression (C=0.1),0.679956


### 3.2. Классификация на 3 бизнес-категории ('bad', 'so-so', 'great')

**Бизнес-контекст:** Для конечного пользователя приложения важнее понимать общий вердикт блюда, а не ловить разницу между 0 и 1 или 4 и 5.

Объединяем целые оценки в 3 бизнес-класса:
- `bad`: оценки 0 и 1 (провальные блюда)
- `so-so`: оценки 2 и 3 (посредственные блюда)
- `great`: оценки 4 и 5 (отличные блюда)

In [62]:
# 1. Функция перевода чисел в 3 бизнес-категории
def map_to_3classes(rating_int):
    if rating_int in [0, 1]:
        return 'bad'
    elif rating_int in [2, 3]:
        return 'so-so'
    else:
        return 'great'

# 2. Создаем новую целевую переменную
y_3class = y_int.apply(map_to_3classes)

# 3. Смотрим распределение новых классов
print("Распределение 3 бизнес-классов:")
print(y_3class.value_counts())
print("\nВ процентах от всей базы:")
print(y_3class.value_counts(normalize=True) * 100)

Распределение 3 бизнес-классов:
rating
great    14475
so-so     1945
bad       1763
Name: count, dtype: int64

В процентах от всей базы:
rating
great    79.607326
so-so    10.696805
bad       9.695870
Name: proportion, dtype: float64


In [63]:
# Делим на train и test со стратификацией по 3 классам
X_train_3c, X_test_3c, y_train_3c, y_test_3c = train_test_split(
    X, y_3class, test_size=0.2, random_state=42, stratify=y_3class
)

# Новый наивный baseline (всегда отвечать самым частым классом)
most_frequent_3c = y_train_3c.mode()[0]
y_pred_naive_3c = np.full_like(y_test_3c, fill_value=most_frequent_3c)

acc_naive_3c = accuracy_score(y_test_3c, y_pred_naive_3c)

print(f"Самый частый класс: {most_frequent_3c}")
print(f"Accuracy наивного классификатора (3 класса): {acc_naive_3c:.4f} ({acc_naive_3c * 100:.2f}%)")

Самый частый класс: great
Accuracy наивного классификатора (3 класса): 0.7960 (79.60%)


#### 3.2.1. Обучение моделей на 3 класса (Метрика: Accuracy)

Проверим, насколько базовые алгоритмы способны обойти наивный baseline (`79.60%`) при делении на 3 класса.
Сравниваем:
1. **Logistic Regression**
2. **Decision Tree Classifier**

In [64]:
models_3c = {
    'Logistic Regression': LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

print(f"Базовый ориентир (Наивная Accuracy 3 класса): {acc_naive_3c:.4f}\n")
print("Результаты на кросс-валидации (5 фолдов, метрика Accuracy):")

for name, model in models_3c.items():
    scores = cross_val_score(
        model, X_train_3c, y_train_3c, 
        cv=5, 
        scoring='accuracy',
        n_jobs=-1
    )
    print(f"- {name}: Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

Базовый ориентир (Наивная Accuracy 3 класса): 0.7960

Результаты на кросс-валидации (5 фолдов, метрика Accuracy):
- Logistic Regression: Accuracy = 0.8048 (+/- 0.0020)
- Decision Tree: Accuracy = 0.8060 (+/- 0.0020)


In [65]:
grid_lr_3c = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid={'C': [0.01, 0.1, 1.0, 5.0]},
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr_3c.fit(X_train_3c, y_train_3c)

best_c_3c = grid_lr_3c.best_params_['C']
print(f"\nЛучший C для 3 классов: {best_c_3c}")
print(f"Accuracy на CV: {grid_lr_3c.best_score_:.4f}")


Лучший C для 3 классов: 0.1
Accuracy на CV: 0.8048


### 3.3. Оптимизация целевой бизнес-метрики: Precision класса 'great'

**Инженерное обоснование:**
В задаче рекомендации питания цена ошибки асимметрична:
- Ошибка I рода (сказать «плохо» на хорошее блюдо): пользователь просто выберет другой рецепт.
- Ошибка II рода (сказать «GREAT» на невкусное блюдо): пользователь потратит время и деньги на несъедобную еду, что разрушит доверие к сервису.

Следовательно, целевой метрикой является **Precision класса 'great'** — процент блюд, которые действительно оказались великолепными среди всех, кому модель присвоила класс `great`.

In [68]:
# Задаем функцию расчета Precision строго для класса 'great'
def precision_great_score(y_true, y_pred):
    return precision_score(y_true, y_pred, labels=['great'], average='micro', zero_division=0)
precision_great_scorer = make_scorer(precision_great_score)
models_precision = {
    'Logistic Regression': LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}
print("Результаты базовых моделей по метрике Precision('great') на кросс-валидации:")
for name, model in models_precision.items():
    scores = cross_val_score(
        model, X_train_3c, y_train_3c,
        cv=5,
        scoring=precision_great_scorer,
        n_jobs=-1
    )
    print(f"- {name}: Precision('great') = {scores.mean():.4f} (+/- {scores.std():.4f})")

Результаты базовых моделей по метрике Precision('great') на кросс-валидации:
- Logistic Regression: Precision('great') = 0.8096 (+/- 0.0022)
- Decision Tree: Precision('great') = 0.8078 (+/- 0.0015)


#### 3.3.1. Исследование ансамблей для максимизации Precision('great')

**Цель:** Применить ансамблевые методы (Random Forest и Gradient Boosting), чтобы повысить точность рекомендаций великолепных блюд (`great`) и минимизировать риск ложноположительных срабатываний.

In [70]:
ensembles_clf = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

In [71]:
print("Результаты ансамблей по Precision('great') на кросс-валидации (5 фолдов):")

ensemble_c_scores = {}
for name, model in ensembles_clf.items():
    scores = cross_val_score(
        model, X_train_3c, y_train_3c,
        cv=5,
        scoring=precision_great_scorer,
        n_jobs=-1
    )
    mean_score = scores.mean()
    ensemble_c_scores[name] = mean_score
    print(f"- {name}: Precision('great') = {mean_score:.4f} (+/- {scores.std():.4f})")

Результаты ансамблей по Precision('great') на кросс-валидации (5 фолдов):
- Random Forest: Precision('great') = 0.8045 (+/- 0.0014)
- Gradient Boosting: Precision('great') = 0.8118 (+/- 0.0025)


#### 3.3.2. Подбор гиперпараметров ансамбля-победителя (GridSearchCV)

Лидером по метрике `Precision('great')` стал **Gradient Boosting** с результатом `0.8118`.  
Оптимизируем его гиперпараметры (`learning_rate` и `max_depth`) с помощью `GridSearchCV` на 5 фолдах кросс-валидации под целевую бизнес-метрику.

In [72]:
param_grid_gb = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

In [73]:
grid_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
    param_grid=param_grid_gb,
    cv=5,
    scoring=precision_great_scorer,
    n_jobs=-1
)

grid_gb.fit(X_train_3c, y_train_3c)

best_gb_params = grid_gb.best_params_
best_gb_score = grid_gb.best_score_

print(f"Лучшие параметры Gradient Boosting: {best_gb_params}")
print(f"Precision('great') на CV после тюнинга: {best_gb_score:.4f}")

Лучшие параметры Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 5}
Precision('great') на CV после тюнинга: 0.8118


#### 3.3.3. Финальная оценка лучших классификаторов на тесте (Precision 'great')

Оцениваем победителей на тестовой выборке `X_test_3c`, замеряя снайперскую точность рекомендаций (`Precision` класса `great`):
1. **Logistic Regression** (`C=0.1`) — лучшая одиночная модель.
2. **Gradient Boosting** (`lr=0.1, depth=5`) — лучший ансамбль.

In [74]:
# 1. Наивный классификатор на тесте
prec_naive = precision_great_score(y_test_3c, y_pred_naive_3c)

# 2. Обучаем лучшую Logistic Regression на train и тестируем
best_lr_final = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
best_lr_final.fit(X_train_3c, y_train_3c)
y_pred_lr_test = best_lr_final.predict(X_test_3c)
prec_lr_test = precision_great_score(y_test_3c, y_pred_lr_test)

# 3. Обучаем лучший Gradient Boosting на train и тестируем
best_gb_final = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
best_gb_final.fit(X_train_3c, y_train_3c)
y_pred_gb_test = best_gb_final.predict(X_test_3c)
prec_gb_test = precision_great_score(y_test_3c, y_pred_gb_test)

In [75]:
results_precision_final = pd.DataFrame({
    'Модель (3 класса)': ['Наивный Baseline', 'Logistic Regression (C=0.1)', 'Gradient Boosting'],
    "Test Precision ('great')": [prec_naive, prec_lr_test, prec_gb_test]
})

results_precision_final

,Модель (3 класса),Test Precision ('great')
0,Наивный Baseline,0.795986
1,Logistic Regression (C=0.1),0.812197
2,Gradient Boosting,0.812696


## 4. Решение (Decision) и Сохранение модели

### Обоснование выбора:
Для продакшн-программы `nutritionist.py` выбирается **классификация на 3 категории** (`bad`, `so-so`, `great`) на базе модели **Gradient Boosting Classifier**, оптимизированной по метрике `Precision('great')`:
1. Регрессия предсказывает непрерывный рейтинг с погрешностью ~1.2 звезды, что не дает однозначного ответа пользователю о качестве блюда.
2. Классификатор оптимизирован под снайперскую точность: он рекомендует категорию `great` только при высокой уверенности (81.3% точности), минимизируя риск испортить ужин ложной рекомендацией.

Обучаем финальную модель на всей выборке и сохраняем её вместе со списком признаков в файл для продакшн-модуля `recipes.py`.

In [77]:
# 1. Обучаем финальную модель Gradient Boosting на всех 18 183 рецептах
final_model = GradientBoostingClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    random_state=42
)
final_model.fit(X, y_3class)
# 2. Формируем словарь артефактов: сама модель + список признаков
model_payload = {
    'model': final_model,
    'features': list(X.columns),
    'classes': list(final_model.classes_)
}
# 3. Сохраняем в файл recipe_model.joblib в папке src
joblib.dump(model_payload, 'recipe_model.joblib')
print("Финал исследования в recipes.ipynb!")
print(f"- Обучено на {X.shape[0]} рецептах и {X.shape[1]} ингредиентах")
print(f"- Модель сохранена в файл: src/recipe_model.joblib")

Финал исследования в recipes.ipynb!
- Обучено на 18183 рецептах и 497 ингредиентах
- Модель сохранена в файл: src/recipe_model.joblib
